# Notebook to Query CV data against JD and find top CVs

In [1]:
# import rt libraries"""
import pandas as pd
import sys
import os
import json

from typing import List





from langfuse.openai import openai
from IPython.display import Markdown



from dotenv import load_dotenv
load_dotenv()

## Langfuse imports
# setup langfuse
from langfuse import Langfuse

langfuse = Langfuse(
  secret_key=os.getenv('LANGFUSE_SECRET_KEY'),
  public_key=os.getenv('LANGFUSE_PUBLIC_KEY'),
  host=os.getenv('LANGFUSE_HOST')
)

In [2]:
%load_ext autoreload
%autoreload 2


In [3]:
## Module imports


sys.path.append(os.path.abspath('..'))
from src.agents import run_jd_extraction_agent, run_resume_extraction_agent, run_evaluator_agent

/home/rishabhs/miniconda3/envs/langfuse/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Initialize Langfuse
from langfuse.langchain import CallbackHandler
 
# Initialize Langfuse CallbackHandler for Langchain (tracing)
langfuse_handler = CallbackHandler()

In [6]:
# Create the standard LangChain configuration block
pipeline_config = {
    "callbacks": [langfuse_handler],
    "run_name": "CV_Evaluation_Batch",
    "tags": ["evaluation_dataset", "llama3"]
}

In [7]:


df_pairs = pd.read_csv('../datasets/jd_resume_pairs.csv')
# rename the first column to 'cv_index'
df_pairs.rename(columns={'Unnamed: 0': 'cv_index'}, inplace=True)



In [8]:
df_pairs.head()

,cv_index,source_jd_title,source_jd_description,assigned_tier,tier_nuance_instruction,generated_resume_text
0,0,Marketing Data Analyst,Title: Sr BI Engineer/ Sr Marketing Data Analy...,Strong Match,Strong Match (Perfect alignment with core skil...,"# Jordan M. Calloway\nDallas, TX | jordan.call..."
1,1,Marketing Data Analyst,Title: Sr BI Engineer/ Sr Marketing Data Analy...,Strong Match,Strong Match (High alignment but leverages alt...,"# Jordan M. Calloway\nDallas, TX | (214) 555-0..."
2,2,Marketing Data Analyst,Title: Sr BI Engineer/ Sr Marketing Data Analy...,Average Match,Average Match (Lacks the target seniority leve...,# Jordan M. Calloway\n📧 jordan.calloway@email....
3,3,Marketing Data Analyst,Title: Sr BI Engineer/ Sr Marketing Data Analy...,Average Match,Average Match (Solid industry context but miss...,"# Jordan M. Calloway\n📍 Nashville, TN (CST) | ..."
4,4,Marketing Data Analyst,Title: Sr BI Engineer/ Sr Marketing Data Analy...,Bad Match,Bad Match (Excellent profile but in a complete...,"# Jordan M. Castellano\nAustin, TX | (512) 448..."


In [9]:
# Test the jd extraction agent
sample_jd = """
Canva is seeking a Senior AI Engineer to join our Melbourne hub. 
In this role, you will be responsible for orchestrating advanced agentic workflows 
using LangGraph and LangChain to power new user-facing features. 
The ideal candidate must have 5+ years of experience in backend development with Python, 
and a deep, hands-on understanding of deploying vector databases like pgvector in production environments. 
Familiarity with AWS cloud infrastructure and CI/CD pipelines is highly desirable but not strictly required.
"""

print("Running JD Extraction Agent (Local Llama 3)...\n")

# Execute the agent
extracted_criteria = run_jd_extraction_agent(sample_jd)

# Pretty-print the resulting dictionary to verify the structure
print(json.dumps(extracted_criteria, indent=4))

Running JD Extraction Agent (Local Llama 3)...

{
    "role_title": "Senior AI Engineer",
    "dense_search_string": "Seeking Senior AI Engineer to orchestrate advanced agentic workflows using LangGraph and LangChain, deploy vector databases like pgvector in production environments with Python backend development experience.",
    "must_have_skills": [
        "Python",
        "LangGraph",
        "LangChain",
        "pgvector"
    ],
    "nice_to_have_skills": [
        "AWS cloud infrastructure",
        "CI/CD pipelines"
    ],
    "years_experience": "5+",
    "core_responsibilities": [
        "Orchestrating advanced agentic workflows",
        "Deploying vector databases in production environments",
        "Powering new user-facing features"
    ]
}


In [18]:
# test with some JDs from the saved dataset
source_jds = df_pairs['source_jd_description'].unique()
for jd in source_jds[:2]:
    print(f'Extracting the key points for the jd - {jd}\n\n')
    # Execute the agent
    extracted_criteria = run_jd_extraction_agent(jd)

    # Pretty-print the resulting dictionary to verify the structure
    print(json.dumps(extracted_criteria, indent=4))
    print("-" * 60)

Extracting the key points for the jd - Title: Sr BI Engineer/ Sr Marketing Data Analyst
Location: Remote. MST, CST, EST preferred. Team is in CST.
Perm or Contract: 6-month contract with the intention for conversion to permanent

Responsibilities • Write complex SQL queries while creating Tableau views to analyze data from various sources. • Gather client reporting and analytics requirements related to financial marketing data. • Represent us proudly in front of client managers, technical sales engineers, and internal stakeholders. • Translate client needs into technical specifications for Tableau reporting. • Create new data stories as needed.

Must Haves: • Adept expertise creating dashboards and generating reports with Tableau. • A deep data analysis background in financial institutions, analyzing marketing data and solutions. • Strong SQL skills with a demonstrated background creating complex queries, views, and database tables. • A demonstrative skill set with the ability to prese

## Test the CV Extraction Agent

In [9]:
sample_anonymized_resume = """
[LOCATION_REDACTED] - Victoria, Australia

PROFESSIONAL SUMMARY
Technology leader with over 17 years of experience in the IT, analytics, and information management sectors. Recently transitioned into an Automation and Insights Manager role within a major financial institution. Passionate about AI development and deploying agentic orchestration frameworks into production.

EXPERIENCE
Automation and Insights Manager | Banking Sector | 2026 - Present
- Architected and deployed production-grade RAG pipelines utilizing LangChain, LangGraph, and pgvector to automate complex regulatory reporting dashboards.
- Led the transition of legacy data systems to modern cloud-native infrastructures.

Senior Data Scientist | 2020 - 2026
- Built predictive models using Python, utilizing advanced vector databases and LLMs to improve customer insights and operational efficiency.
- Managed heavy system configurations and local deployment pipelines exclusively via Fedora Linux terminals.
"""

print("Running Resume Extraction Agent (Local Llama 3)...\n")

# Execute the agent
extracted_profile = run_resume_extraction_agent(sample_anonymized_resume)

# Pretty-print the resulting dictionary
print(json.dumps(extracted_profile, indent=4))

Running Resume Extraction Agent (Local Llama 3)...

{
    "total_years_experience": "17 years",
    "technical_skills": [
        "Python",
        "LangChain",
        "LangGraph",
        "pgvector",
        "Fedora Linux"
    ],
    "domain_knowledge": [
        "Regulatory Reporting (RAG)",
        "Cloud-Native Infrastructures",
        "Agentic Orchestration Frameworks",
        "AI Development"
    ],
    "key_achievements": [
        "Deployed production-grade RAG pipelines utilizing LangChain, LangGraph, and pgvector to automate complex regulatory reporting dashboards.",
        "Transitioned legacy data systems to modern cloud-native infrastructures.",
        "Built predictive models using Python, utilizing advanced vector databases and LLMs to improve customer insights and operational efficiency."
    ]
}


## Testing the Evaluator Agent

In [16]:
sample_jd = """
Canva is seeking a Senior AI Engineer to join our Melbourne hub. 
In this role, you will be responsible for orchestrating advanced agentic workflows 
using LangGraph and LangChain to power new user-facing features. 
The ideal candidate must have 5+ years of experience in backend development with Python, 
and a deep, hands-on understanding of deploying vector databases like pgvector in production environments. 
Familiarity with AWS cloud infrastructure and CI/CD pipelines is highly desirable but not strictly required.
"""

sample_anonymized_resume = """
[LOCATION_REDACTED] - Victoria, Australia

PROFESSIONAL SUMMARY
Technology leader with over 17 years of experience in the IT, analytics, and information management sectors. Recently transitioned into an Automation and Insights Manager role within a major financial institution. Passionate about AI development and deploying agentic orchestration frameworks into production.

EXPERIENCE
Automation and Insights Manager | Banking Sector | 2026 - Present
- Architected and deployed production-grade RAG pipelines utilizing LangChain, LangGraph, and pgvector to automate complex regulatory reporting dashboards.
- Led the transition of legacy data systems to modern cloud-native infrastructures.

Senior Data Scientist | 2020 - 2026
- Built predictive models using Python, utilizing advanced vector databases and LLMs to improve customer insights and operational efficiency.
- Managed heavy system configurations and local deployment pipelines exclusively via Fedora Linux terminals.
"""

In [17]:
print("1. Extracting Job Requirements...")
jd_json = run_jd_extraction_agent(sample_jd, config=pipeline_config)

print("2. Extracting Candidate Profile...")
cv_json = run_resume_extraction_agent(sample_anonymized_resume, config=pipeline_config)

print("3. Running Final Evaluator Agent...\n")
final_evaluation = run_evaluator_agent(jd_json, cv_json, config=pipeline_config)

# Pretty-print the final decision matrix
print("=== FINAL EVALUATION MATRIX ===")
print(json.dumps(final_evaluation, indent=4))

1. Extracting Job Requirements...
2. Extracting Candidate Profile...
3. Running Final Evaluator Agent...

=== FINAL EVALUATION MATRIX ===
{
    "match_score": 92,
    "recommendation": "Strong Hire",
    "strengths": [
        "17 years of experience exceeds the required 5+ years.",
        "Proficient in Python, LangGraph, LangChain, and pgvector, meeting all must-have skills.",
        "Domain knowledge in Agentic Orchestration Frameworks aligns with core responsibilities."
    ],
    "critical_gaps": [],
    "justification": "The candidate's extensive experience and expertise in the required technologies make them an ideal fit for the Senior AI Engineer role. Their achievements demonstrate a strong understanding of agentic workflows, vector databases, and cloud-native infrastructures, which directly align with the core responsibilities. The only minor gap is the lack of AWS cloud infrastructure experience, but their overall profile more than compensates for this."
}


In [12]:
## Test against a poorly matching CV
sample_anonymized_resume = """
# [LOCATION_REDACTED]\n[EMAIL_REDACTED] | [PHONE_NUMBER_REDACTED] | linkedin.com/in/[CANDIDATE_NAME]\n\n---\n\n## Summary\n\n
# Senior Marketing Data Analyst and BI Engineer with 9+ years of experience delivering data-driven insights within financial services and fintech environments. Proven track record designing complex Tableau dashboards and translating ambiguous business requirements into actionable reporting solutions for executive and client-facing audiences. 
# Deep background in financial marketing analytics — including campaign performance, customer acquisition funnels, and multi-channel attribution — paired with advanced SQL proficiency across relational database platforms. 
# Recognized for bridging the gap between technical teams and non-technical stakeholders, consistently driving client satisfaction and measurable reporting adoption.\n\n---\n\n## Core Competencies\n\n- Tableau Dashboard Design & Development (Tableau Desktop, Tableau Server, Tableau Prep)\n- Complex SQL Query Authoring — Views, CTEs, Stored Procedures, Joins, Window Functions\n- Financial Marketing Data Analysis — Campaign ROI, Customer Segmentation, LTV, Attribution\n- Client Requirements Gathering & Stakeholder Management\n- Executive-Level Data Presentation & Data Storytelling\n- DOMO Reporting & SSRS Report Development\n- Jira & Confluence — Agile Project Tracking & Documentation\n- Wireframing & Process Diagramming — Lucidchart, Visio\n- Data Modeling & Relational Database Design\n- Cross-Functional Collaboration — Sales Engineering, Marketing Operations, Finance\n\n---\n\n## Professional History\n\n**Senior BI Engineer / Marketing Data Analyst**\n*Veritas Financial Solutions — [LOCATION_REDACTED]*\n*March 2020 – Present*\n\n- Design and maintain a library of 40+ Tableau dashboards used by C-suite, regional bank managers, and external client stakeholders to track marketing campaign performance, lead conversion rates, and customer acquisition costs across digital and direct mail channels.\n- Write complex SQL queries — including multi-table joins, recursive CTEs, and window functions — against a SQL Server data warehouse housing 200M+ marketing and transactional records to support ad hoc analysis and scheduled reporting.\n- Lead client-facing requirements gathering sessions with marketing directors and technical sales engineers at regional bank and credit union clients, translating business objectives into formal technical specifications for dashboard development.\n- Developed a standardized Tableau reporting framework for financial marketing clients that reduced new dashboard build time by 35% and improved first-delivery approval rate to 91%.\n- Produce SSRS paginated reports distributed to compliance and finance stakeholders on a weekly and monthly cadence, ensuring regulatory alignment with marketing spend disclosures.\n- Built supplementary reporting views in DOMO for a national mortgage lender client, integrating CRM and paid media data to surface channel-level ROAS in near real-time.\n- Maintain project documentation, sprint tracking, and requirement tickets in Jira and Confluence, coordinating delivery timelines across analytics, engineering, and client success teams.\n- Present quarterly business reviews and data stories to VP- and Director-level client contacts, consistently receiving high satisfaction scores (avg. 4.7/5.0 CSAT) across 12+ client accounts.\n\n**Marketing Data Analyst**\n*Centennial Bank Group — [LOCATION_REDACTED]*\n*August 2016 – February 2020*\n\n- Supported the retail banking marketing team by building Tableau workbooks to visualize campaign performance metrics including email open rates, branch foot traffic lift, and new account acquisition by segment.\n- Authored SQL queries and database views in SQL Server to extract, cleanse, and model marketing data from core banking systems, CRM platforms, and third-party data vendors.\n- Partnered with marketing managers to define KPIs for product launch campaigns (checking, savings, HELOC), establishing baseline benchmarks and reporting cadences.\n- Created Lucidchart workflow diagrams to document data pipelines and dashboard refresh logic, improving onboarding time for new analysts by approximately 40%.\n- Delivered monthly executive reporting packages to the CMO and VP of Retail Banking, synthesizing multi-channel marketing performance into narrative-driven slide decks.\n- Assisted in migration of legacy Crystal Reports output to SSRS, standardizing 25+ operational marketing reports and reducing manual formatting effort by 60%.\n\n**Junior Data Analyst**\n*Apex Marketing Analytics — [LOCATION_REDACTED]*\n*June 2014 – July 2016*\n\n- Supported senior analysts in building SQL-based data extracts and Excel reporting models for financial services clients including insurance carriers and community banks.\n- Assisted with Tableau dashboard prototyping and data validation across client reporting environments.\n- Documented client requirements and meeting notes in Confluence; tracked deliverables using Jira.\n\n---\n\n## Education\n\n**Bachelor of Science, Management Information Systems**\n*University — [LOCATION_REDACTED]*\n*Graduated*\n\n---\n\n## Certifications & Professional Development\n\n- Tableau Desktop Specialist — Tableau (Salesforce), *Issued 2021*\n- Tableau Desktop Certified Associate — Tableau (Salesforce), *Issued 2022*\n- Microsoft Certified: Data Analyst Associate (Power BI) — *Issued 2020*\n- Google Analytics Individual Qualification — *Issued 2023*
"""

In [15]:
print("1. Extracting Job Requirements...")
jd_json = run_jd_extraction_agent(sample_jd, config=pipeline_config)

print("2. Extracting Candidate Profile...")
cv_json = run_resume_extraction_agent(sample_anonymized_resume, config=pipeline_config)

print("3. Running Final Evaluator Agent...\n")
final_evaluation = run_evaluator_agent(jd_json, cv_json, config=pipeline_config)

# Pretty-print the final decision matrix
print("=== FINAL EVALUATION MATRIX ===")
print(json.dumps(final_evaluation, indent=4))

1. Extracting Job Requirements...
2. Extracting Candidate Profile...
3. Running Final Evaluator Agent...

=== FINAL EVALUATION MATRIX ===
{
    "match_score": 60,
    "recommendation": "Borderline",
    "strengths": [
        "9+ years of experience aligns with the '5+' requirement.",
        "Technical skills include SQL, which is a nice-to-have skill."
    ],
    "critical_gaps": [
        "Missing Python and LangGraph/LangChain/pgvector skills in technical skills list.",
        "No experience mentioned in key achievements related to orchestrating advanced agentic workflows or deploying vector databases in production environments."
    ],
    "justification": "The candidate has extensive experience, but lacks critical skills for the role. While they have some relevant technical skills, they are not aligned with the must-have requirements. The candidate's achievements demonstrate strong reporting and data analysis capabilities, but do not directly relate to the core responsibilities 